# Phase 6 (companion) — loop-back: do the 4 ceiling-raising levers help?

> Issue #27 · result written up in `backend/docs/ml/07-train.md` §4b. **Runnable but not
> auto-executed** (~2h CPU). Test fold stays SEALED — VAL only.

Phase-6 baseline = val AUROC **0.684** (big/topk), below the 0.85 target. Before deciding
Phase-7 entry we try **all four** obvious levers, **reporting all** (not just a winner) to bound
val-overfitting:

1. **longer schedule** (200 epochs vs 60)
2. **ENU `x_rel/y_rel`** added (path-structure features → target `zone_violation`, our worst type)
3. **per-phase** arrival/departure models (tighter manifold → subtle altitude)
4. **per-type characterisation** (which types we miss)

**Spoiler (the finding):** none beat baseline within noise (±0.013); ENU & per-phase make it
worse; ENU does **not** fix `zone_violation`. The per-type read shows the 0.68 is an
**architecture property, not under-tuning** — the AE caps on subtle *spatial* anomalies (which
belong to the geofence Layer-3) and excels on *dynamic* ones (loiter 0.95, intercept 0.83). See
07-train.md §4b.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

REPO = Path.cwd()
while not (REPO / "backend/core/preprocessing.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from backend.core import split as sp
from backend.core.features import apply_segment_derivations
from backend.core.geo import LEMD_LAT, LEMD_LON
from backend.core.preprocessing import (AE_FEATURES, SCALER_FEATURES, make_scaler,
                                         to_sequences, to_sequences_loss_mask)
from backend.core.inject import make_eval_set, inject_segment, assign_kinds, INJECTION_KINDS
from backend.core import lstm_ae as ae

M = REPO / "backend/models/phase6"; SEED = 42; MPDLAT = 111_320.0
EP_LONG, EP_STD = 200, 60
BASE = dict(hidden=64, latent=32, num_layers=2)
OURTYPES = list(INJECTION_KINDS)

clean_df = pd.read_parquet(M/"clean_df.parquet"); meta = pd.read_parquet(M/"meta.parquet")
split = sp.split_by_monday(meta); sp.assert_firewall(split)
train_df = sp.subset(clean_df, split.train_ids); val_df = sp.subset(clean_df, split.val_ids)
T = int(np.percentile(train_df.groupby("segment_id").size(), 95))
print(f"T={T}  train_seg={len(split.train_ids)}  val_seg={len(split.val_ids)}")

In [ ]:
# helpers
def add_enu(df):
    df = df.copy()
    df["x_rel"] = (df["lon"]-LEMD_LON)*MPDLAT*np.cos(np.radians(df["lat"]))
    df["y_rel"] = (df["lat"]-LEMD_LAT)*MPDLAT
    return df

def window(df, T, scaler, feat_list, scaler_list):
    sc = [feat_list.index(c) for c in scaler_list]; rows, ids = [], []
    for sid, seg in df.groupby("segment_id", sort=False):
        seg = seg.sort_values("time"); f = seg[feat_list].to_numpy("float64").copy()
        f[:, sc] = scaler.transform(seg[scaler_list]); f = f[:T]
        if len(f) < T: f = np.vstack([f, np.zeros((T-len(f), len(feat_list)))])
        rows.append(f.astype("float32")); ids.append(sid)
    return (np.stack(rows) if rows else np.empty((0,T,len(feat_list)),"float32")), ids

def build_eval_frames(clean_df, ids, seed, inject_rate=0.5, onset=0.5):
    rng = np.random.default_rng(seed); ids = list(ids); rng.shuffle(ids)
    inj = set(ids[:int(round(len(ids)*inject_rate))])
    kinds = dict(zip(sorted(inj), assign_kinds(len(inj), rng))) if inj else {}
    sub = clean_df[clean_df["segment_id"].isin(set(ids))]; frames, y, kind = [], [], []
    for sid, seg in sub.groupby("segment_id", sort=False):
        if sid in inj:
            f,_ = inject_segment(seg, kinds[sid], rng, onset_fraction=onset)
            frames.append(f); y.append(1); kind.append(kinds[sid])
        else:
            frames.append(seg.sort_values("time").reset_index(drop=True)); y.append(0); kind.append("normal")
    return frames, np.array(y), kind

def per_type_auroc(y, score, kind):
    kind = np.array(kind); norm = score[y==0]; out = {}
    for k in OURTYPES:
        sel = kind==k
        if sel.sum()==0: continue
        out[k] = round(float(roc_auc_score(np.r_[np.zeros(len(norm)),np.ones(sel.sum())], np.r_[norm,score[sel]])),4)
    out["OVERALL"] = round(float(roc_auc_score(y,score)),4)
    return out

results = {}
scaler9 = make_scaler().fit(train_df[SCALER_FEATURES])
vs = make_eval_set(clean_df, split.val_ids, scaler9, T, seed=SEED, fold="val", inject_rate=0.5)
base_model = ae.load_checkpoint(str(M/"lstm_ae_best.pt"))
results["0_baseline_big_topk"] = per_type_auroc(vs.y, ae.reconstruction_error(base_model, vs.X, vs.loss_mask, agg="topk"), vs.kind)
ev_frames, ev_y, ev_kind = build_eval_frames(clean_df, split.val_ids, SEED)
ev_df = pd.concat(ev_frames, ignore_index=True)
results["0_baseline_big_topk"]

In [ ]:
# (1) longer schedule (200 ep)
X9_tr,_,_ = to_sequences(train_df, T, scaler9); m9_tr = to_sequences_loss_mask(train_df, T)
X9_vn,_,_ = to_sequences(val_df, T, scaler9);   m9_vn = to_sequences_loss_mask(val_df, T)
m_long,h_long = ae.train_autoencoder(X9_tr,m9_tr,X9_vn,m9_vn,lr=1e-3,max_epochs=EP_LONG,patience=20,batch_size=128,seed=SEED,**BASE)
results["1_longer_schedule"] = per_type_auroc(vs.y, ae.reconstruction_error(m_long,vs.X,vs.loss_mask,agg="topk"), vs.kind)
results["1_longer_schedule"]["_best_epoch"] = h_long.best_epoch
results["1_longer_schedule"]

In [ ]:
# (4) ENU x_rel/y_rel added (11 features) — same anomalies as vs (same seed)
FEAT11 = AE_FEATURES + ["x_rel","y_rel"]; SCAL11 = SCALER_FEATURES + ["x_rel","y_rel"]
tr_enu = add_enu(train_df); scaler11 = make_scaler().fit(tr_enu[SCAL11])
X11_tr,_ = window(tr_enu,T,scaler11,FEAT11,SCAL11); m11_tr = to_sequences_loss_mask(tr_enu,T)
vn_enu = add_enu(val_df);   X11_vn,_ = window(vn_enu,T,scaler11,FEAT11,SCAL11); m11_vn = to_sequences_loss_mask(vn_enu,T)
ev_enu = add_enu(ev_df);    X11_ev,_ = window(ev_enu,T,scaler11,FEAT11,SCAL11); m11_ev = to_sequences_loss_mask(ev_enu,T)
m_enu,_ = ae.train_autoencoder(X11_tr,m11_tr,X11_vn,m11_vn,lr=1e-3,max_epochs=EP_STD,patience=8,batch_size=128,seed=SEED,**BASE)
results["4_enu_xy"] = per_type_auroc(ev_y, ae.reconstruction_error(m_enu,X11_ev,m11_ev,agg="topk"), ev_kind)
results["4_enu_xy"]

In [ ]:
# (3) per-phase arrival/departure models
def phase_of(seg):
    a = seg.sort_values("time")["baroaltitude"].to_numpy(); return "dep" if a[-1] > a[0] else "arr"
ph_train = train_df.groupby("segment_id").apply(phase_of, include_groups=False)
models_ph = {}
for ph in ("arr","dep"):
    tdf = train_df[train_df["segment_id"].isin({s for s in split.train_ids if ph_train.get(s)==ph})]
    Xt,_,_ = to_sequences(tdf,T,scaler9); mt = to_sequences_loss_mask(tdf,T)
    models_ph[ph],_ = ae.train_autoencoder(Xt,mt,max_epochs=EP_STD,patience=8,batch_size=128,seed=SEED,**BASE)
ev_phase = [phase_of(f) for f in ev_frames]; s_ph = np.zeros(len(ev_y))
for ph in ("arr","dep"):
    idx = [i for i,p in enumerate(ev_phase) if p==ph]
    if not idx: continue
    fdf = pd.concat([ev_frames[i] for i in idx], ignore_index=True)
    Xp,_ = window(fdf,T,scaler9,AE_FEATURES,SCALER_FEATURES); mp = to_sequences_loss_mask(fdf,T)
    s_ph[idx] = ae.reconstruction_error(models_ph[ph],Xp,mp,agg="topk")
results["3_per_phase"] = per_type_auroc(ev_y, s_ph, ev_kind)
results["3_per_phase"]

In [ ]:
pd.DataFrame(results).T

## Conclusion

None of the four levers beats baseline within noise (CI ±0.013); ENU and per-phase are *worse*,
and ENU does **not** lift `zone_violation`. The 0.684 is an **architecture/benchmark property,
not under-tuning**: the AE caps on subtle *spatial* anomalies (1–3 km lateral shifts blend into
the wide normal LEMD route cloud — it detects them fine at 20–80 km, see notebook 10) and excels
on *dynamic* anomalies (loiter, intercept). Small zone/position violations belong to the geofence
rules layer (D-008 Layer 3); the AE earns its keep on the dynamic anomalies the rules can't catch.
**Baseline big/topk stands.** Full write-up: `backend/docs/ml/07-train.md` §4b.